# Lab 12 - RAG Data Prep: Chunking, Overlap, and Metadata

**Week 2 - Data Engineering for LLM Pipelines**

You will turn a small document corpus into a high-quality RAG index input by implementing **three chunkers** (sentence-window, heading-aware, token-bounded), attaching **provenance-rich metadata**, and validating **JSONL shards** line by line. A recurring theme: clean text *without* destroying the punctuation that carries meaning.

### Outcomes
By the end of this lab you will be able to:

1. Implement sentence-window, heading-aware, and token-bounded chunkers with configurable **overlap**.
2. Attach consistent **metadata** (source, section, timestamps, schema version) to each chunk.
3. Clean text so whitespace is normalized but **meaning-bearing punctuation** survives.
4. Emit **JSONL shards** for embedding and retrieval, and validate them line by line.
5. Compare chunk-length distributions and choose sensible defaults for a project.

> **This lab is self-contained.** The setup cells generate their own synthetic corpus and train their own tiny tokenizer. Nothing from a prior lab is required.

## Prerequisites and Setup

- **Python 3.13** with `regex`, `orjson`, `numpy`, `pandas`, `tqdm`, and `tokenizers`.
- Everything below runs from a clean directory. The first setup cell creates the working folders; the second writes a synthetic corpus under `data/text`.

Run the setup cells top to bottom, then work through Parts A through D in order. Each part ends with a `check(...)` call so you can self-assess before moving on.

In [ ]:
%pip install --upgrade regex orjson tqdm tokenizers

In [ ]:
from pathlib import Path
import json, hashlib, itertools
from datetime import datetime, timezone
from typing import List, Iterator, Dict, Any

import regex as re
import orjson
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

for _p in ["artifacts/tokenizer", "artifacts/rag", "artifacts/stats"]:
    Path(_p).mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# Generate a small, self-contained synthetic corpus (fictional "Cordwell Home & Hardware").
# No external files or prior-lab artifacts are required.
from pathlib import Path

DOCS = {
"returns_policy.md": """# Cordwell Home & Hardware - Returns Policy

## Overview
Cordwell accepts returns within 90 days of purchase for most items. A valid receipt
or order number (e.g., CW-2025-004512) is required. Refunds are issued to the original
payment method within 5-7 business days.

## Non-Returnable Items
Custom-cut lumber, clearance tools, and opened paint (tinted) are final sale. Was the
item opened? If yes, a 15% restocking fee may apply. Special orders over $1,234.50 need
manager approval before a refund is processed.

## Pro Desk Accounts
Pro Desk members receive extended 180-day returns. Contact the Pro Desk at ext. 4412 or
email prodesk@cordwell.example for bulk return authorizations.
""",
"deckpro_spec.md": """# DeckPro 2000 Cordless Drill - Specification Sheet

## Power and Battery
The DeckPro 2000 ships with a 20V MAX lithium-ion battery. Peak torque is 650 in-lbs.
Runtime is approximately 45 minutes under continuous load. Charge time: 60 minutes to 80%.

## Pricing and Availability
Retail price is $129.99. Pro Desk price is $114.49 with a valid account. Ships in 2-3
business days. SKU: DP2000-BLK. Item weight is 3.2 lbs.

## Warranty
Covered by a 3-year limited warranty. Does the warranty cover the battery? Yes - the
battery carries a separate 1-year warranty. Register within 30 days at cordwell.example.
""",
"api_ratelimits.md": """# Cordwell Inventory API - Rate Limits and Errors

## Section 2.1: Rate Limits
The Inventory API allows 30 requests per minute per API key. Authenticated clients get a
higher core limit of 5,000 requests per hour. Exceeding the limit returns HTTP 429 with a
Retry-After header (in seconds).

## Section 2.2: Error Codes
A 404 means the SKU was not found. A 409 indicates a stock conflict (concurrent write).
Use exponential backoff: 1s, 2s, 4s, 8s, 16s. Do not retry on 4xx except 429.

## Section 2.3: Pagination
Results are paginated at 100 items per page. Use the cursor token from the response;
do not construct page numbers manually.
""",
"store_ops.txt": """STORE OPERATIONS MANUAL - AISLE RESET

Aisle resets happen weekly. Reset the plumbing aisle (Aisle 12) every Monday before 8:00 AM.
Verify shelf tags match the planogram version 3.4. Report mismatches to the department lead.

SAFETY NOTES

Wet-floor signage is mandatory near the paint mixing station. Forklift operation requires
certification renewed every 3 years. Report near-misses within 24 hours using form OPS-17.
""",
}

CORPUS_ROOT = Path("data/text")
CORPUS_ROOT.mkdir(parents=True, exist_ok=True)
for _name, _body in DOCS.items():
    (CORPUS_ROOT / _name).write_text(_body, encoding="utf-8")

print("Corpus files:", sorted(p.name for p in CORPUS_ROOT.glob("*")))

### Self-check helpers

Run the next cell once. It defines the `check_*` functions used throughout the lab. They test **properties** of your output (does overlap exist, is the budget respected, is metadata complete), not exact string matches, so there is room for reasonable implementation differences.

In [ ]:
# Self-check helpers - run this cell once, do not edit.
# Each check returns (passed, message). They test properties, not exact output.
"""Soft self-assessment checks for Lab 12. Each returns (bool, message)."""

def _ok(msg="OK"): return (True, msg)
def _no(msg): return (False, msg)

def check_normalize(fn):
    try:
        out = fn("  Price  was   $1,234.50\ton 2025-01-03 \n\n  Yes! ")
    except NotImplementedError:
        return _no("normalize_text not implemented yet")
    if "  " in out:
        return _no("multiple consecutive spaces remain; collapse runs of spaces")
    for token in ["$1,234.50", "2025-01-03", "Yes!"]:
        if token not in out:
            return _no(f"meaning-bearing token lost: {token!r}")
    if out != out.strip():
        return _no("leading/trailing whitespace not stripped")
    if "\t" in out:
        return _no("tabs should be collapsed to spaces")
    return _ok("normalize_text preserves semantics and collapses whitespace")

def check_sentences(to_sentences, chunk_by_sentences):
    text = ("The first sentence describes the returns policy in full detail. "
            "The second sentence explains the restocking fee for opened items. "
            "The third sentence covers Pro Desk extended return windows clearly. "
            "The fourth sentence lists the non-returnable clearance categories. "
            "The fifth sentence closes with the manager approval threshold rule.")
    try:
        sents = to_sentences(text)
        chunks = chunk_by_sentences(text, window=3, overlap=1)
    except NotImplementedError:
        return _no("sentence functions not implemented yet")
    if len(sents) < 4:
        return _no(f"expected >=4 sentences, got {len(sents)}")
    if len(chunks) < 2:
        return _no("expected multiple overlapping windows")
    shared = sents[2]
    if not (shared in chunks[0] and shared in chunks[1]):
        return _no("windows do not overlap by the requested amount")
    return _ok(f"{len(sents)} sentences, {len(chunks)} overlapping windows")

def check_headings(chunk_by_headings):
    text = ("# Title A\n" + "word " * 200 + "\n## Title B\n" + "word " * 200)
    try:
        chunks = chunk_by_headings(text, max_chars=400, overlap_chars=80)
    except NotImplementedError:
        return _no("chunk_by_headings not implemented yet")
    if len(chunks) < 2:
        return _no("large sections should split into multiple chunks")
    if any(len(c) > 400 for c in chunks):
        return _no("a chunk exceeds max_chars")
    if chunks[0][-80:] not in chunks[1][:160]:
        return _no("adjacent chunks from a split do not overlap")
    return _ok(f"{len(chunks)} chunks, max_chars respected, overlap present")

def check_tokens(chunk_by_tokens, token_len):
    text = " ".join(f"word{i}" for i in range(400))
    try:
        chunks = chunk_by_tokens(text, max_tokens=60, overlap_tokens=15)
    except NotImplementedError:
        return _no("chunk_by_tokens not implemented yet")
    if len(chunks) < 3:
        return _no("expected several token-bounded chunks")
    over = [c for c in chunks if token_len(c) > 60]
    if over:
        return _no(f"{len(over)} chunk(s) exceed the max_tokens budget")
    return _ok(f"{len(chunks)} chunks all within token budget")

def check_record(build_one):
    try:
        rec = build_one("data/text/x.md", "Some example chunk text here.", "tokens", 0)
    except NotImplementedError:
        return _no("record builder not implemented yet")
    for k in ("doc_id", "chunk_id", "text", "metadata"):
        if k not in rec:
            return _no(f"record missing top-level key: {k}")
    for k in ("source", "schema_version", "created_at", "chunk_index", "n_chars", "chunker"):
        if k not in rec["metadata"]:
            return _no(f"metadata missing key: {k}")
    if not rec["chunk_id"].startswith(rec["doc_id"]):
        return _no("chunk_id should start with doc_id")
    return _ok("record has stable ids and complete provenance metadata")

def check_validator(validate_jsonl, tmp_path="_checktmp.jsonl"):
    import json, os
    good = {"text": "x" * 50, "metadata": {"source": "s"}}
    bad_short = {"text": "tiny", "metadata": {"source": "s"}}
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(good) + "\n")
        f.write(json.dumps(bad_short) + "\n")
        f.write("{not valid json\n")
    try:
        total, bad = validate_jsonl(tmp_path)
    except NotImplementedError:
        return _no("validate_jsonl not implemented yet")
    finally:
        os.remove(tmp_path)
    if total != 3:
        return _no(f"expected total=3, got {total}")
    if bad != 2:
        return _no(f"expected bad=2 (short + malformed), got {bad}")
    return _ok("validator counts total and flags short/malformed lines")


## Part A - Cleaners that Preserve Meaning

In RAG, punctuation frequently changes meaning: decimals, hyphenated ranges, colons in section numbers, and question marks all matter. A naive cleaner that strips every non-word character would flatten `$1,234.50` and `2025-01-03` into meaningless noise.

Implement `normalize_text` so it collapses redundant whitespace but leaves punctuation, digits, and symbols intact. Keep newlines as soft boundaries so the heading detector in Part B can still see them.

In [ ]:
def normalize_text(s: str) -> str:
    """Collapse runs of whitespace while PRESERVING meaning-bearing characters.

    Contract:
      - Remove carriage returns.
      - Collapse runs of tabs/spaces to a single space.
      - Trim spaces around newlines, but KEEP newlines (soft boundaries).
      - Strip leading/trailing whitespace.
      - Do NOT remove punctuation, digits, currency, or symbols.
    """
    s = s.replace("\r", "")
    s = re.sub(r"[\t ]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = s.strip("\n ")
    return s

In [ ]:
raw = "Price was  $1,234.50 on 2025-01-03\n\n\nSee Section 2.1:  Rate Limits? Yes!"
print(normalize_text(raw))
print()
print(check_normalize(normalize_text))

## Part B - Three Chunkers

Each strategy answers "where should a chunk end?" differently:

- **Sentence-window** slides a window of N sentences with an overlap of K sentences. Good when meaning lives at the sentence level.
- **Heading-aware** aligns chunks to document structure, then windows any oversized section. Good for manuals and specs.
- **Token-bounded** grows a chunk until it hits a token budget, then overlaps by a token count. This is what most embedding models actually care about.

### B1 - Sentence-window chunker

Implement `to_sentences` (a lightweight regex splitter, no ML models) and `chunk_by_sentences` (a sliding window with overlap).

In [ ]:
# Split on whitespace that follows sentence-ending punctuation and precedes a capital/digit.
_SENT_SPLIT = re.compile(r"(?<=\S[.!?])\s+(?=[A-Z0-9])")

def to_sentences(text: str) -> List[str]:
    """Return a list of sentences. Very short fragments (<40 chars) are merged
    into the previous sentence so splitter noise does not create tiny chunks."""
    text = normalize_text(text)
    sents = _SENT_SPLIT.split(text)
    merged: List[str] = []
    for s in sents:
        s = s.strip()
        if not s:
            continue
        if merged and len(s) < 40:
            merged[-1] += " " + s
        else:
            merged.append(s)
    return merged

def chunk_by_sentences(text: str, window: int = 5, overlap: int = 2) -> List[str]:
    """Sliding window over sentences. Consecutive chunks share `overlap` sentences."""
    if window <= overlap:
        raise ValueError("window must be greater than overlap")
    sents = to_sentences(text)
    out: List[str] = []
    i = 0
    while i < len(sents):
        end = min(i + window, len(sents))
        out.append(" ".join(sents[i:end]))
        if end == len(sents):
            break
        i = end - overlap
    return out

print(check_sentences(to_sentences, chunk_by_sentences))

### B2 - Heading-aware chunker

Implement `split_by_headings` (break at heading lines, then re-attach tiny orphan sections) and `chunk_by_headings` (window any section that exceeds `max_chars`, overlapping by `overlap_chars`).

In [ ]:
# A heading is a Markdown "#..." line OR an ALL-CAPS line (a common plaintext convention).
_HEADING = re.compile(r"^(?:\s*#+\s+.+|\s*[A-Z][A-Z0-9 .:&-]{3,}$)", re.M)

def split_by_headings(text: str, min_section_chars: int = 300) -> List[str]:
    """Split text at heading boundaries. Sections smaller than `min_section_chars`
    are appended to the previous section so headings are not orphaned."""
    text = normalize_text(text)
    starts = [m.start() for m in _HEADING.finditer(text)]
    if not starts:
        return [text]
    sections: List[str] = []
    pos = 0
    for start in starts:
        if start > pos:
            sections.append(text[pos:start].strip("\n "))
        pos = start
    sections.append(text[pos:].strip("\n "))
    merged: List[str] = []
    for sec in sections:
        if not sec:
            continue
        if merged and len(sec) < min_section_chars:
            merged[-1] += "\n" + sec
        else:
            merged.append(sec)
    return merged

def chunk_by_headings(text: str, max_chars: int = 600, overlap_chars: int = 150) -> List[str]:
    """Emit heading-aligned sections, sliding a character window (with overlap)
    across any section longer than `max_chars`."""
    if max_chars <= overlap_chars:
        raise ValueError("max_chars must be greater than overlap_chars")
    out: List[str] = []
    for section in split_by_headings(text):
        start = 0
        while start < len(section):
            end = min(start + max_chars, len(section))
            out.append(section[start:end])
            if end == len(section):
                break
            start = end - overlap_chars
    return out

print(check_headings(chunk_by_headings))

### B3 - Token-bounded chunker

Real embedders think in tokens, not characters. First train a tiny byte-level BPE tokenizer on the corpus (the training boilerplate is provided). Then implement `token_len` and `chunk_by_tokens`.

> **Engineer-to-engineer:** think of this like packing fixed-size network frames. You fill a frame up to its budget, then re-send a little of the tail in the next frame so nothing straddling the boundary is lost.

In [ ]:
# Train a tiny byte-level BPE tokenizer on the corpus itself (self-contained,
# no downloads). In production you would load your model's real tokenizer instead.
_texts = [p.read_text(encoding="utf-8") for p in sorted(Path("data/text").glob("*"))]

tok = Tokenizer(BPE(unk_token="<unk>"))
tok.pre_tokenizer = ByteLevel(add_prefix_space=False)
_trainer = BpeTrainer(vocab_size=1500, min_frequency=1,
                      special_tokens=["<unk>", "<pad>", "<s>", "</s>"])
tok.train_from_iterator(_texts, trainer=_trainer)
tok.save("artifacts/tokenizer/bytebpe.json")
print("Trained tokenizer, vocab size:", tok.get_vocab_size())

def token_len(s: str) -> int:
    """Number of tokens the tokenizer assigns to `s`."""
    return len(tok.encode(s).ids)

def chunk_by_tokens(text: str, max_tokens: int = 120, overlap_tokens: int = 24) -> List[str]:
    """Greedily grow word windows until the next word would exceed `max_tokens`,
    then step back so each chunk overlaps the prior one by ~`overlap_tokens`.
    A single word longer than the budget is emitted as its own chunk."""
    if max_tokens <= overlap_tokens:
        raise ValueError("max_tokens must be greater than overlap_tokens")
    words = normalize_text(text).split()
    out: List[str] = []
    i, n = 0, len(words)
    while i < n:
        cur: List[str] = []
        j = i
        while j < n:
            if cur and token_len(" ".join(cur + [words[j]])) > max_tokens:
                break
            cur.append(words[j])
            j += 1
        out.append(" ".join(cur))
        if j >= n:
            break
        back = 0
        while back < len(cur) - 1 and token_len(" ".join(cur[len(cur) - back - 1:])) < overlap_tokens:
            back += 1
        next_i = j - back
        i = next_i if next_i > i else i + 1
    return out

print(check_tokens(chunk_by_tokens, token_len))

## Part C - Build Chunks, Metadata, and Shards

A chunk without provenance is hard to trust or debug in production. Attach metadata to every chunk (where it came from, which strategy produced it, when, and its position), give it stable ids, and stream records to size-bounded JSONL shards.

Implement `build_one` (assemble one record) and `write_shards` (chunk every document and write shards).

In [ ]:
def iter_documents() -> Iterator[tuple]:
    """Yield (source_path, raw_text) for every file in data/text/."""
    for path in sorted(Path("data/text").glob("*")):
        yield str(path), path.read_text(encoding="utf-8", errors="ignore")

CHUNKERS = {
    "sentences": (chunk_by_sentences, {"window": 5, "overlap": 2}),
    "headings":  (chunk_by_headings,  {"max_chars": 600, "overlap_chars": 150}),
    "tokens":    (chunk_by_tokens,    {"max_tokens": 120, "overlap_tokens": 24}),
}

# Choose the chunker whose shards you will build in Part C.
CHUNKER = "tokens"
SCHEMA_VERSION = "rag-chunk-v2"
print("Active chunker:", CHUNKER, "with", CHUNKERS[CHUNKER][1])

In [ ]:
def build_one(source: str, text: str, chunker: str, j: int) -> Dict[str, Any]:
    """Assemble one RAG record: stable ids + provenance-rich metadata."""
    doc_id = hashlib.sha1(source.encode("utf-8")).hexdigest()[:16]
    return {
        "doc_id": doc_id,
        "chunk_id": f"{doc_id}-{j:04d}",
        "text": text,
        "metadata": {
            "source": source,
            "schema_version": SCHEMA_VERSION,
            "created_at": datetime.now(timezone.utc).isoformat(),
            "chunk_index": j,
            "n_chars": len(text),
            "chunker": chunker,
        },
    }

def write_shards(chunker: str, max_shard_bytes: int = 50_000_000) -> List[str]:
    """Chunk every document and stream records to size-bounded JSONL shards."""
    fn, params = CHUNKERS[chunker]
    idx = cur = 0
    files: List[str] = []
    path = f"artifacts/rag/rag_{chunker}_{idx:03d}.jsonl"
    out = open(path, "w", encoding="utf-8"); files.append(path)
    for source, text in tqdm(iter_documents(), desc=f"chunk[{chunker}]"):
        chunks = fn(normalize_text(text), **params)
        for j, ch in enumerate(chunks):
            rec = build_one(source, ch, chunker, j)
            line = orjson.dumps(rec).decode() + "\n"
            b = len(line.encode("utf-8"))
            if cur + b > max_shard_bytes:
                out.close(); idx += 1; cur = 0
                path = f"artifacts/rag/rag_{chunker}_{idx:03d}.jsonl"
                out = open(path, "w", encoding="utf-8"); files.append(path)
            out.write(line); cur += b
    out.close()
    return files

shard_files = write_shards(CHUNKER)
print("Wrote shards:", shard_files)
print(check_record(build_one))

In [ ]:
files = sorted(Path("artifacts/rag").glob(f"rag_{CHUNKER}_*.jsonl"))
total_lines = sum(1 for p in files for _ in open(p, encoding="utf-8"))
print("SHARDS:", [str(p) for p in files])
print("TOTAL CHUNKS:", total_lines)
print("-" * 60)
with open(files[0], encoding="utf-8") as f:
    for line in itertools.islice(f, 2):
        rec = json.loads(line)
        print(rec["chunk_id"], "| n_chars:", rec["metadata"]["n_chars"])
        print(rec["text"][:180], "...")
        print()

## Part D - Validate, Compare, and Choose Defaults

Never trust a shard you have not validated. Implement `validate_jsonl`, then compare all three strategies on the same corpus and pick defaults.

In [ ]:
def validate_jsonl(path: str, min_chars: int = 40) -> tuple:
    """Return (total, bad). A line is bad if it is not valid JSON, is missing
    `text` or `metadata`, or its text is shorter than `min_chars`."""
    total = bad = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            total += 1
            try:
                obj = json.loads(line)
                ok = (isinstance(obj, dict) and "text" in obj and "metadata" in obj
                      and len(obj["text"]) >= min_chars)
                if not ok:
                    bad += 1
            except Exception:
                bad += 1
    return total, bad

for p in files:
    t, b = validate_jsonl(str(p))
    print(f"{p.name}: total={t} bad={b}")
print(check_validator(validate_jsonl))

In [ ]:
# Rebuild all three strategies and compare their chunk-length distributions.
rows = []
for name in CHUNKERS:
    fs = write_shards(name)
    lens = []
    tot = badtot = 0
    for p in fs:
        t, b = validate_jsonl(p); tot += t; badtot += b
        with open(p, encoding="utf-8") as f:
            lens += [len(json.loads(line)["text"]) for line in f]
    rows.append({
        "chunker": name, "shards": len(fs), "chunks": tot, "bad": badtot,
        "mean_chars": round(float(np.mean(lens)), 1),
        "p50_chars": float(np.percentile(lens, 50)),
        "p90_chars": float(np.percentile(lens, 90)),
    })

comparison = pd.DataFrame(rows)
comparison.to_csv("artifacts/stats/chunker_comparison.csv", index=False)
comparison

In [ ]:
# Sample token counts from the active chunker's shards to sanity-check the budget.
sample = []
for p in sorted(Path("artifacts/rag").glob(f"rag_{CHUNKER}_*.jsonl"))[:1]:
    with open(p, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 200:
                break
            sample.append(token_len(json.loads(line)["text"]))

print({
    "mean_tokens": round(float(np.mean(sample)), 1),
    "p95_tokens": float(np.percentile(sample, 95)),
    "max_tokens": int(np.max(sample)),
})

## Wrap-Up

Answer these in a markdown cell:

1. Which chunker and parameters would you choose for this corpus, and why?
2. Which metadata fields are essential for your retriever (for example `source`, `section`, `created_at`)?
3. What were your validator results (bad over total)? What rule would you tighten before production?
4. If a downstream embedder required sentence boundaries, how would you adapt your token-bounded chunker?

**Common pitfalls:** over-aggressive cleaning that kills punctuation; zero overlap causing context loss at boundaries; missing metadata; shards written but never validated.

## Instructor Currency Notes

- `datetime.now(timezone.utc)` is the correct timezone-aware call. `datetime.utcnow()` is deprecated in Python 3.12 and later and returns a naive datetime - do not use it.
- `hashlib.sha1` is used here only for **content addressing** (a stable id), not for security. That use is fine; it is not a cryptographic claim.
- `orjson.dumps(...)` returns `bytes`; `.decode()` turns it into a `str` for line writing. Do not wrap it in `json.dumps` again.
- The tiny in-corpus tokenizer is for teaching token-counting only. In production, load the **exact tokenizer of your embedding or generation model** so token budgets match what the model sees.
- In pandas 3.0 the comparison DataFrame's text columns use the new `str` dtype, not `object`. `select_dtypes(include=["object"])` would return nothing for them.